Steps of this program
 1. Read a page 
 2. use recursivetextsplitter to format data
 3. convert the doc to embeddings and storing them in FAISS DB
 4. Create a stuff doc chain, create a retrieval chain from the FAISS DB
 5. Create a retrieval chain using the FAISS retrevier and document chain


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.prompts import ChatPromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_classic.chains import create_retrieval_chain

import os
from dotenv import load_dotenv
load_dotenv()


True

In [10]:

loader = WebBaseLoader("https://www.bahria.edu.pk/Home/AcademicRoadmapDetails?roadmapId=46")

docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter()
document = text_splitter.split_documents(docs)

In [15]:
len(document)

3

In [ ]:
docs[0].page_content

'\n\n\n\n\n\n\nBahria University\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBahria University\nDiscovering Knowledge\n\n\n\n\n\n\n\n\n\n\nAbout\nAcademics\nAdmissions\nResearch & Innovations\nInternational\nCampus\n\n\n\n\n\n\n\n\nBS Information Technology\n\n\nImportant for all candidates\n\nApplicant for admission MUST MEET THE ELIGIBILITY REQUIREMENTS set-forth by Bahria University.\nCandidates are advised to CONFIRM THEIR ELIGIBILITY prior apply.\nIn case of annual system, eligibility will be determined on the basis of result in percentages.\nIn case of semester system, eligibility will be determined on the basis of CGPA obtained out of 4.00.\nIn case the result is shown both in CGPA and percentage, CGPA will be considered.\n\nEligibility Criteria\nMinimum 50% marks in Intermediate (HSSC) examination (*Pre-Medical/Pre-Engg.) or equivalent qualification with Mathematics certified by IBCC. *Pre-Medical students are required to pass 02 deficiency courses of mathematics during the first year of 

In [11]:

llm = ChatGoogleGenerativeAI(api_key=os.getenv("GOOGLE_API_KEY"), temperature=0, model="gemini-3.5-flash")

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [13]:
vector = FAISS.from_documents(document, embeddings)

In [16]:
prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>

    Question: {input}
"""
)



In [17]:
document_chain = create_stuff_documents_chain(llm, prompt)

In [18]:
retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [23]:
response = retrieval_chain.invoke({
    "context": "You are the admin who is going to tell the name of the courses of BS information technology",
    "input":"What are the courses name of semester 1 only"
})

print(response['answer'])

Based on the provided context, the course names for Semester 1 are:

1. Introduction to Information & Communication Technology
2. Introduction to Information & Communication Technology Lab
3. Computer Programming
4. Computer Programming Lab
5. Applied Physics
6. Applied Physics Lab
7. *Islamic Studies /** Ethics
8. Discrete Mathematics
9. Professional Practices and Ethics
10. **Tajweed
